# 🚂 TARDIS Project — TRAINING THE PREDICTION MODEL

***

### The Research Team :

* **Raphaël Permentier**
* **Stan Gaumain**
* **Come Chaslerie**

***

### Our Mission :

**We now have a cleaned dataset (see `tardis_eda.ipynb`), so we can build a regression model to predict the average arrival delay of a train.**

***

### Plan of the notebook :

* **Step 1 :** Load every library needed
* **Step 2 :** Store the cleaned dataset to use it
* **Step 3 :** Choose the target and the features
* **Step 4 :** Split the dataset into training and test
* **Step 5 :** Preprocessing pipeline (one-hot encoding for the strings)
* **Step 6 :** Create the function to compare the results with the metrics
* **Step 7 :** Find the baseline that predicts the mean to have a reference
* **Step 8 :** Create the three models to compare
* **Step 9 :** Compare the models to find the best
* **Step 10 :** Tune the best model with GridSearchCV
* **Step 11 :** Compare before and after tuning
* **Step 12 :** Save the best model

***


## 📦 **Step 1 : Load every library we need :**

***

### **Why do we use each library :**

* **pandas** : manipulates CSV files as a table (DataFrame).
* **numpy** : uses maths on arrays.
* **matplotlib** : useful for charts and graphics.
* **seaborn** : goes with matplotlib, gives a better rendering.
* **joblib** : used to save our model to be able to use it (ex: in the dashboard) without redoing everything.
* **scikit-learn** : the only authorized machine learning lib in Python by the subject.


In [ ]:
# pandas and numpy to manipulate the dataset
import numpy as np
import pandas as pd

# matplotlib for the charts
import matplotlib.pyplot as plt

# joblib to save the model into a file
import joblib

# train_test_split = split the data / GridSearchCV = hyperparameter tuning
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# parallel_config : lets us switch the joblib backend from "loky" (subprocess workers)
# to "threading" (same process). Threading is needed when we want our warning filters
# to apply during GridSearchCV with n_jobs=-1. Only useful when running the (commented) grid.fit() below.
from joblib import parallel_config

# ColumnTransformer : apply different transformations to different columns
from sklearn.compose import ColumnTransformer

# OneHotEncoder : transform string columns into 0/1 columns
from sklearn.preprocessing import OneHotEncoder

# Pipeline : chains the preprocessing step and the model into one single object.
# You call .fit() once and it fits both ; you call .predict() once and the input
# gets preprocessed then passed to the model automatically. Makes saving and reloading easy.
from sklearn.pipeline import Pipeline

# DummyRegressor : baseline model that predicts the mean
from sklearn.dummy import DummyRegressor

# regression models we use
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# regression metrics we use : MAE, MSE (for RMSE) and R2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 📂 **Step 2 : Load the cleaned dataset :**

***

**`tardis_eda.ipynb` already cleaned the dataset for us to use so we just get it from the `cleaned_dataset.csv`.**

**The separation between the cleaning and the modeling is a good practice because each notebook has its job, and there is no mix between them.**


In [ ]:
# read the cleaned dataset CSV given by tardis_eda.ipynb into a DataFrame with the pandas lib
df = pd.read_csv("cleaned_dataset.csv")
# print the shape (rows, columns) to know the size of the dataframe
print("Shape :", df.shape)
# show the first 5 rows to see if everything looks good
df.head()

**The dataset has 10034 rows and 16 columns. Every row is one combination (departure station / arrival station / month), and the columns contain infos about the trains.**


## 🎯 **Step 3 : Pick the target and the features :**

***

### **What is the difference between target and features ?**

**In machine learning we have two main things :**

* **The target (`y`) :** what we want to predict. There is only one target column.
* **The features (`X`) :** all the information the model is given to make its prediction. There can be many feature columns.

**The model creates a function such as `f(X) ≈ y`. In our case the function will try to estimate the average arrival delay.**

***

### **Our Target :**

**We predict `Average delay of all trains at arrival`. It is a number in minutes, so we are doing a *regression* problem.**

***

### **Our Features :**

**We carefully picked the columns depending on their relevance (Ex: the `Route` is a very precise indicator because it identifies a whole line in a single value).**

**Categorical features (strings, will be one-hot encoded) :**

* **`Service`** : if it's National or International
* **`Departure station`** and **`Arrival station`** : the two endpoints of the trip
* **`Route`** : combination of departure + arrival, captures the typical delay of a whole line

**Numeric features (used directly) :**

* **`Average journey time`** : how long the trip is supposed to last
* **`Month`** and **`Year`** : temporal features added during the cleaning
* **`Cancellation_Rate`** : how often trains on this line get cancelled
* **`Internal_Fault_pct`** and **`External_Fault_pct`** : breakdown of the causes of disruptions
* **`real_nb_train`** : real number of trains that actually ran on this line


In [ ]:
# the column we want to predict
TARGET = "Average delay of all trains at arrival"

# categorical columns (strings) -> we will rework them (one-hot encode) to be usable for our model
FEATURES_CAT = ["Service", "Departure station", "Arrival station", "Route"]

# numeric columns -> we can use them directly
FEATURES_NUM = [
    "Average journey time",
    "Month",
    "Year",
    "Traffic_Pressure",
    "Cancellation_Severity",
    "Hight_Delay_Impact",
    "Medium_Delay_Impact",
    "Light_Delay_Impact",
    "Internal_Fault_pct",
    "External_Fault_pct",
    "Delay_Probability",
]

# final list of features used by the model
FEATURES = FEATURES_CAT + FEATURES_NUM

# copy the features and target columns into another dataframe
data = df[FEATURES + [TARGET]].copy()

# show the shape after filtering
print("Shape :", data.shape)
data.head()

## ✂️ **Step 4 : Split the data in training and test :**

***

### **Why don't we train and test on the same data :**

**If we check if a model works with the same data it was trained on, the score won't represent how the model truly works. It would just memorize the training set (*overfitting*). What we really want to know is : how good is the model on an unknown dataset ?**

**To check that well, we split the dataset into two parts :**

* **Training set (80%)** : the model learns from this part. It can see the features AND the target.
* **Test set (20%)** : the model never sees this dataset during the training. We use it at the end to measure the real performance.

***

### **Why we always fix the `random_state` :**

**`train_test_split` mixes the rows randomly before splitting. If we don't fix the seed, each run produces a different split and different scores. With a fixed seed (`42` here), the split always takes the same rows, so every model is evaluated on the same test set and the comparison between models is fair.**

***

### **Why 80/20 :**

**Standard practice. The training set needs to be big enough for the model to learn, but at the same time the test set must be big enough for the score to be meaningful. 80/20 is the usual percentage for big datasets.**


In [ ]:
X = data[FEATURES]  # the features the model is allowed to look at
y = data[TARGET]  # the target we want the model to predict

# split 80% training / 20% test with a fixed seed.
X_training, X_test, y_training, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% of the rows for the test set
    random_state=42,  # fixed seed
)

# print the shapes to make sure the split worked
print("Train :", X_training.shape, " Test :", X_test.shape)

## ⚙️ **Step 5 : Preprocessing pipeline :**

***

### **What and why ? :**

**Scikit-learn models can only use numbers. But `Service`, `Departure station`, `Arrival station` and `Route` are strings (`"PARIS MONTPARNASSE"`, `"LE MANS"`, etc.). Sending these directly to a `LinearRegression` would crash the model.**

***

### **How ? : one-hot encoding**

**One-hot encoding turns each string column into columns of 0 or 1, one per value. Example with the `Service` column :**

```
Service                               Service_NATIONAL   Service_INTERNATIONAL
NATIONAL                                      1                    0
INTERNATIONAL                                 0                    1
```

**Each row has a `1` in one and only one column and `0` in the others. The model can now treat each station / service as numbers. The downside is that with around 100 different stations we get 100 columns, but it works and it is easy to use.**

***

### **Why a `ColumnTransformer` :**

**The string columns need one-hot encoding, but the numeric ones don't. The `ColumnTransformer` lets us apply DIFFERENT transformations to DIFFERENT columns in one shot.**

***

### **Why we also wrap it in a `Pipeline` later :**

**When the preprocessor is bundled with the model inside a `Pipeline`, we get two big benefits :**

* **Same encoding everywhere** : the same `ColumnTransformer` is applied on train, on test, and later on user inputs from the dashboard. No risk of forgetting to encode somewhere.
* **One single file to save** : `joblib.dump(pipeline, "model.joblib")` saves preprocessing AND model together. The dashboard just does `joblib.load("model.joblib")` and gets a working object straight away, no encoding to redo by hand.


In [ ]:
preprocessor = (
    ColumnTransformer(  # apply different transformations to different columns
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                FEATURES_CAT,
            ),  # one-hot encoding (transform strings into 0/1 columns)
            ("num", "passthrough", FEATURES_NUM),  # nothing to do, we just put as it is
        ]
    )
)

## 📏 **Step 6 : Evaluation metrics :**

***

**Here is a small function `evaluate()`, that will train the model on the training data, predict on the test data, computes our three metrics, and stores every useful infos for the comparison.**

***

### **Understanding the three regression metrics :**

#### **1. MAE — Mean Absolute Error**

**Formula :** `MAE = mean(|y_real - y_predicted|)`

**What it does :** for each test sample, take the average of each absolute difference between the prediction and the real value.

**Unit :** here, **minutes**. So `MAE = 2.5` means : on average, our prediction is wrong by 2.5 minutes.

**Pros :** easy to interpret. "We are off by X minutes on average."

**Cons :** treats every error the same. The size of the error is proportional to the difference, big mistakes are not that visible.

**Lower is better.**

***

#### **2. RMSE — Root Mean Squared Error**

**Formula :** `RMSE = sqrt(mean((y_real - y_predicted)²))`

**What it does :** like MAE but each error is squared. Then we take the square root to come back to the original values.

**Unit :** here, **minutes** too.

**Why squared :** the square penalizes big errors more than small errors. Which completes the MAE metric.

**Cons :** hard to understand and very sensitive to huge errors.

**Lower is better.**

***

#### **3. R² — Coefficient of determination**

**Formula :** `R² = 1 - (sum of squared errors of the model) / (sum of squared errors of a "always predict the mean" model)`

**What it does :** measures how much better the model is than just predicting the average all the time.

**Range :**

* **`R² = 1`** : perfect model, predicts everything exactly right.
* **`R² = 0`** : the model is as good as predicting the mean.
* **`R² < 0`** : the model is even worse than predicting the mean.

**Unit :** doesn't have a unit, can be compared with any other R².

**Interpretation :** `R² = 0.4` means the model explains 40% of the natural variance of the delays. The other 60% comes from things we did not include in our features (weather, accidents, etc.). The closer R² is to 1, the more of the variability we are capturing.

**Higher is better.**

***

#### **Bonus : R² standard deviation**

**A single R² on the test set gives us one number, but it depends on the rows stored in the test set. To know how *stable* this score really is, we also run a quick 5-fold cross-validation on the training data and compute the standard deviation of the R² across the folds.**

**Reading rule :**

* **Small std (< 0.05)** : the model gives consistent scores no matter which fold is used → reliable.
* **Big std (> 0.1)** : the model is fragile, its quality depends a lot on the data it sees → less reliable.

**The recap line printed by `evaluate()` shows it as `R² = 0.426 ± 0.012`.**


In [ ]:
# list that stores one dictionary per model, easy to turn into a DataFrame after
results = []


# function that trains a model and stores its scores
def evaluate(name, model, X_training, y_training, X_test, y_test):
    # 1) train the model on the training data
    model.fit(X_training, y_training)
    # 2) predict on the test data
    y_pred = model.predict(X_test)
    # 3) compute the three metrics on the test set
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    # 4) compute the R2 standard deviation across 5 cross-validation folds
    r2_scores = cross_val_score(
        model, X_training, y_training, cv=5, scoring="r2", n_jobs=-1
    )
    r2_std = r2_scores.std()
    # 5) save the scores in the results list
    results.append(
        {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2, "R2_std": r2_std}
    )
    # 6) print a recap line
    print(
        f"{name:s} | MAE = {mae:.3f} | RMSE = {rmse:.3f} | R2 = {r2:.3f} ± {r2_std:.3f}"
    )

## 🥱 **Step 7 : Baseline (predict the mean) :**

***

### **Why start with a baseline :**

**It's interesting to have a baseline so we can compare our model, if the model doesn't beat the baseline, then something went wrong. So we need to know the metrics of the baseline first.**

**A baseline is a very simple prediction strategy. We use `DummyRegressor(strategy="mean")` which gets the average delay from the training set, ignoring all the inputs. It's very stupid and dull but that's the point.**

***

### **This gives us :**

* **A reference :** if a trained model gives MAE = 3 minutes and the baseline also gives MAE = 3 minutes, then our model is badly trained and should be corrected.
* **A base for the metrics :** R² should be 0 for the baseline (by definition). We use this model to find R² for the other models and if the value is higher than 1 or a lot lower than 0, this means that something went wrong for our model.


In [ ]:
baseline = DummyRegressor(strategy="mean")  # predicts the mean of y_training
evaluate(
    "Baseline (mean)", baseline, X_training, y_training, X_test, y_test
)  # train and score it with the helper, this is the first row of the results table

**Result :** The R² of the baseline is 0 because it does not take the inputs into account. So our models have to beat this score.


## 🧠 **Step 8 : Our three models :**

***

### **📈 Linear Regression :**

#### **What a Linear Regression does :**

**It finds the weights of the params (`w1, w2, ..., wn`) and gets `b` to minimize the prediction error :**

```
predicted_delay = w1 * feature_1 + w2 * feature_2 + ... + wn * feature_n + b
```

#### **Why it is a good basic model :**

* **Fast :** training takes a less than a second.
* **Simple :** the math is quite easy, multiplications and additions inside a fixed formula (one formula gives the optimal weights directly).
* **Interpretable :** you can understand each weight and why they got their values. The greater the weight, the heavier the feature has importance for the final result.
* **Can beat some good models in certain situations :** if a random forest barely beats a linear regression, then the linear regression is better (a lot faster).

#### **What it cannot do :**

**Linear regression comes from a linear relationships between the features and the results. So if our dataset is non-linear (e.g. "trains in winter from a specific station tend to be very late"), it won't have good results. That's why we try other models.**


In [ ]:
# Pipeline : the one-hot encoding is donne first, then the LinearRegression
# receives the numeric output and fits its weights. Calling .fit() on the pipeline does both.
linreg = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", LinearRegression()),
    ]
)
# train and score it with the function
evaluate("Linear Regression", linreg, X_training, y_training, X_test, y_test)

**Result :** The linear regression beats the baseline, so it's better than nothing and it learns at least a small part of the patterns. But the relations between features and result are in our case not linear, so another model should have better results.


***

### **🌲 Decision Tree :**

#### **How a Decision Tree works :**

**It asks yes/no questions on the features to split the data into smaller groups, and gets the average delay in each group (leaf). Example :**

```
Is journey_time < 60 ?
├── Yes → Is number_of_trains < 200 ?
│         ├── Yes → predict 1.5 min delay
│         └── No  → predict 3.2 min delay
└── No  → Is month in [12, 1, 2] (winter) ?
          ├── Yes → predict 6.8 min delay
          └── No  → predict 4.1 min delay
```

#### **Why could it beat the linear regression :**

**It can manage *non-linear* relations (ex: "long trips in winter from Paris are extra bad").**

#### **The danger : overfitting**

**With a `max_depth` too high, the tree will keep splitting data until each leaf contains only one value. The problem is that it will simply memorize the training data but will perform very bad on the test data.**

**We start with `max_depth=10` to limit overfitting and have a first score, then the hyperparameter tuning step will look for a better depth.**


In [ ]:
# pipeline : preprocess + decision tree
tree = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=5,  # each tree can go up to 10 levels of yes/no questions
                random_state=42,
            ),
        ),
    ]
)

# train + score
evaluate("Decision Tree", tree, X_training, y_training, X_test, y_test)

**Result :** With `max_depth=10` the tree gets a positive R² (around 0.23), so it learns something, but it stays weaker than the Linear Regression and far behind the Random Forest. A single tree alone is unstable — the next model (Random Forest) fixes that, and the tuning step will also try different depths.


***

### **🌳 Random Forest :**

#### **What is a Random Forest ?**

**A Random Forest is a *bag* (collection) of many Decision Trees (the model just used before). Each tree is trained on :**

* A **random subset of the rows** called *bootstrap sampling*.
* A **random subset of the features** at each split.

**To find a result, every tree gives its own prediction and the forest takes the average of them.**

#### **Why does it beat a single tree :**

**A decision tree is unstable and unpredictable : change one row in the training set and the results could look completely different. Random Forest prevents this : each tree makes some mistakes, but taking the average of all of them minimizes that instability, so only the repetitive behavior is shown. This is called the *wisdom of the crowd* effect.**

#### **Settings used here :**

* **`n_estimators=400`** : 400 trees in the forest. More trees = generally better results but slower. 400 is a good starting point.
* **`max_depth=10`** : each tree can go up to 10 levels of yes/no questions.
* **`n_jobs=-1`** : use every CPU core in parallel to train the trees, much faster.


In [ ]:
# pipeline : preprocess + random forest
rf = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,  # number of trees in the forest
                max_depth=5,  # each tree can go up to 10 levels of yes/no questions
                n_jobs=-1,  # train trees in parallel on every CPU core
                random_state=42,
            ),
        ),
    ]
)

# train + score
evaluate("Random Forest", rf, X_training, y_training, X_test, y_test)

**Result :** The Random Forest already beats every other model without any tuning. Getting the average of all the trees prevents the overfit seen on the decision tree and gives a better predictions.


## 📊 **Step 9 : Compare the models :**

***

### **What we are looking for :**

**The "best" model, with the lowest MAE / RMSE and the maximum R² on the test set.**

**Usually the three metrics go in the same way but it's good to still check the three of them.**


In [ ]:
# build the comparison table from the results list
results_df = pd.DataFrame(results).set_index("Model")
results_df

In [ ]:
# function that returns colors : green for the best, gray for the others
def colors_for(series, lower_is_better):
    # for MAE / RMSE the best is the minimum, for R2 it is the maximum
    best = series.idxmin() if lower_is_better else series.idxmax()
    return ["#2ecc71" if i == best else "#bdc3c7" for i in series.index]


# three bar charts side by side, one per metric
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# MAE : lower is better
results_df["MAE"].plot(
    kind="bar",
    ax=axes[0],
    color=colors_for(results_df["MAE"], True),
    title="MAE (lower = better)",
    edgecolor="black",
)

# RMSE : lower is better
results_df["RMSE"].plot(
    kind="bar",
    ax=axes[1],
    color=colors_for(results_df["RMSE"], True),
    title="RMSE (lower = better)",
    edgecolor="black",
)

# R2 : higher is better
results_df["R2"].plot(
    kind="bar",
    ax=axes[2],
    color=colors_for(results_df["R2"], False),
    title="R2 (higher = better)",
    edgecolor="black",
)

# write the value on top of each bar
for ax in axes:
    for p in ax.patches:
        ax.annotate(
            f"{p.get_height():.2f}",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

**Result :** The Random Forest is the best model on every metric : it beats the Linear Regression, the single Decision Tree and the baseline. Now we will try to tune it with `GridSearchCV` to see if we can have an even higher score.


## 🔧 **Step 10 : Tune the Random Forest with GridSearchCV :**

***

### **Hyperparameters vs Parameters :**

**There are two kinds of values inside a model :**

* **Parameters** : automatically found with `.fit(...)` (ex: the weights of the linear regression, the splits of each tree).
* **Hyperparameters** : chosen by us before training (ex: `max_depth`, `n_estimators`).

**Tuning means trying a lot of combinations of hyperparameters to find the best one.**

***

### **What `GridSearchCV` does :**

1. **Grid :** we provide a list of values for each hyperparameter, and it tests every combination (cartesian product of the lists).
2. **CV (Cross-Validation) :** for each combination, the training data is split into K equal parts (folds). The model is trained on K-1 folds and scored on the remaining one. We repeat K times (each fold gets to be the validation set once) and average the K scores. This gives a much more reliable estimate of the model quality than evaluating once on a single split — random luck on one fold gets averaged out.
3. **Best :** at the end it tells us which combination of hyperparameters got the best average CV score, and gives us a fully retrained model with those hyperparameters.

***

### **Quick vs Slow tuning :**

**To show the difference between a basic tuning and a big one, we tune the Random Forest TWICE :**

1. **Quick tuning** : a small grid (8 combinations × 5 folds = 40 fits). Takes around 1 minute.
2. **Slow tuning** : a big grid (1280 combinations × 5 folds = 6400 fits). Takes several hours.

**Then we compare the two : time taken, best parameters found, and score on the test set. This shows whether the extra hours are really worth it or not.**


In [ ]:
# Quick tuning : small grid that runs fast
quick_grid = {
    "model__n_estimators": [100, 200],  # number of trees
    "model__max_depth": [10, 15],  # depth of each tree
    "model__min_samples_leaf": [1, 5],  # min samples in a leaf
    "model__min_samples_split": [2],  # min samples needed to split a node
    "model__max_features": ["sqrt"],  # number of features tested at each split
}

# total : 8 combinations x 5 folds = 40 fits -> roughly 1 minute
quick_search = GridSearchCV(
    estimator=rf,  # the pipeline we want to tune
    param_grid=quick_grid,  # the values to test
    cv=5,  # 5-fold cross-validation
    scoring="neg_mean_absolute_error",  # sklearn convention : score is maximized, so MAE is negated
    n_jobs=-1,  # use every CPU core in parallel
    verbose=0,  # no progress output (set to 1 or 2 to see fits in real time)
)

# Uncomment the two lines below to actually run the quick tuning (~1 minute).
# We keep them commented because we already know the best params (see hardcoded model below).

# quick_search.fit(X_training, y_training)
# quick_model = quick_search.best_estimator_


# print the best hyperparameters found by the GridSearch

# print("Best hyperparameters :")
# for param, value in quick_search.best_params_.items():
#     print(f"  {param} = {value}")

**Quick tuning result :** the small grid finished in less than a 1 minute. We got a first combination of hyperparameters, but we could find a better one with a wider search.


In [ ]:
# Slow tuning : big grid that explores more combinations
param_grid = {
    "model__n_estimators": [200, 400, 600, 800],  # number of trees
    "model__max_depth": [10, 15, 20, 30],  # depth of each tree
    "model__min_samples_leaf": [1, 2, 5, 10, 20],  # min samples in a leaf
    "model__min_samples_split": [
        2,
        5,
        10,
    ],  # min samples needed to split a node (min value = 2 in sklearn)
    "model__max_features": [
        "sqrt",
        "log2",
        0.5,
        1.0,
    ],  # number of features tested at each split
}

# total : 4 x 4 x 5 x 3 x 4 = 960 combinations x 5 folds = 4800 fits -> several hours
grid = GridSearchCV(
    estimator=rf,  # the pipeline we want to tune
    param_grid=param_grid,  # the values to test
    cv=5,  # 5-fold cross-validation
    scoring="neg_mean_absolute_error",  # sklearn convention : score is maximized, so MAE is negated
    n_jobs=-1,  # use every CPU core in parallel
    verbose=0,  # no progress output during the fits
)

# Uncomment the two lines below to actually run the slow tuning (several hours).
# We keep them commented because the best params are already hardcoded just below.

# grid.fit(X_training, y_training)
# best_model = grid.best_estimator_   # retrained on the full train set with the best params


# print the best hyperparameters found by the GridSearch

# print("Best hyperparameters :")
# for param, value in grid.best_params_.items():
#     print(f"  {param} = {value}")

**When we ran our hyperparameters finder, we got these values for each tuning :**

***

### **Quick tuning :**

**Finishes in a couple of minutes, gives a decent score quickly.**

**Best hyperparameters quick tuning :**

* **`model__max_depth`** = 15
* **`model__max_features`** = sqrt
* **`model__min_samples_leaf`** = 1
* **`model__min_samples_split`** = 2
* **`model__n_estimators`** = 200

***

### **Slow tuning :**

**Takes a lot longer but greatly improves the results.**

**Best hyperparameters slow tuning :**

* **`model__max_depth`** = 10
* **`model__max_features`** = 0.5
* **`model__min_samples_leaf`** = 2
* **`model__min_samples_split`** = 2
* **`model__n_estimators`** = 400


**Result of the slow tuning :** the slow grid explored 960 combinations × 5 folds = 4800 fits. It found a deeper tree (`max_depth=30`) with `max_features=0.5` and 600 estimators, which gives a clear improvement over the quick tuning. Both tuned models are now stored, so we can compare them.


In [ ]:
# We hardcode the best model with the two set of hyperparameters so we don't have to re-run the tuning each time.

# pipeline with the best hyperparameters found by the quick tuning
quick_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                max_features="sqrt",
                min_samples_leaf=1,
                min_samples_split=2,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

# pipeline with the best hyperparameters found by the slow tuning
best_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=400,
                max_depth=10,
                max_features=0.5,
                min_samples_leaf=2,
                min_samples_split=2,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

# train + score both pipelines
evaluate(
    "Random Forest (quick tuning)", quick_model, X_training, y_training, X_test, y_test
)
evaluate(
    "Random Forest (slow tuning)", best_model, X_training, y_training, X_test, y_test
)

# rebuild and display the comparison table including the two new rows
results_df = pd.DataFrame(results).set_index("Model")

### 📊 **Comparison : quick tuning vs slow tuning :**

**This is the result on the test set of both tunings, side by side. The interesting question : does the slow tuning (much more time spent) give meaningfully better scores than the quick one ?**


In [ ]:
# compare quick tuning vs slow tuning
compare_tuning = results_df.loc[
    ["Random Forest (quick tuning)", "Random Forest (slow tuning)"]
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
# blue = quick tuning, purple = slow tuning
palette = ["#3498db", "#9b59b6"]

for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    bars = ax.bar(
        compare_tuning.index, compare_tuning[metric], color=palette, edgecolor="black"
    )
    direction = "lower" if metric != "R2" else "higher"
    ax.set_title(f"{metric} ({direction} = better)")
    ax.tick_params(axis="x", rotation=15)
    for b in bars:
        ax.annotate(
            f"{b.get_height():.3f}",
            (b.get_x() + b.get_width() / 2, b.get_height()),
            ha="center",
            va="bottom",
            fontsize=10,
        )

plt.tight_layout()
plt.show()

## 🆚 **Step 11 : Before vs after tuning :**

***

**To clearly show the effect of the tuning we compare the default Random Forest with the tuned one.**

**Gray = before tuning, green = after tuning. We want green to be lower than gray for MAE / RMSE, and higher for R².**


In [ ]:
# compare untuned Random Forest vs the best tuned one (slow tuning)
compare = results_df.loc[["Random Forest", "Random Forest (slow tuning)"]]

# three bar charts side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
# gray = before tuning, green = after tuning
palette = ["#bdc3c7", "#2ecc71"]

for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    bars = ax.bar(compare.index, compare[metric], color=palette, edgecolor="black")
    direction = "lower" if metric != "R2" else "higher"
    ax.set_title(f"{metric} ({direction} = better)")
    ax.tick_params(axis="x", rotation=15)
    for b in bars:
        ax.annotate(
            f"{b.get_height():.3f}",
            (b.get_x() + b.get_width() / 2, b.get_height()),
            ha="center",
            va="bottom",
            fontsize=10,
        )

plt.tight_layout()
plt.show()

**Result :** The tuned model has better predictions with a smaller error range, so the model became more reliable.


## 💾 **Step 12 : Save the best model :**

***

### **Why do we save the pipeline and not just the model :**

**The pipeline contains two steps : the `ColumnTransformer` (one-hot encoding) AND the actual regressor. We need to have both the columns and the regressor, or the dashboard would have to re-implement the encoding.**

**By saving the whole pipeline with `joblib.dump(...)`, the dashboard only needs to do :**

```python
import joblib
model = joblib.load("model.joblib")
prediction = model.predict(some_dataframe_with_raw_columns)
```

In [ ]:
# we ignore the baseline and the quick tuning because it does not really learn anything
candidates = results_df.drop(index="Baseline (mean)")
candidates = candidates.drop(index="Random Forest (quick tuning)")

# pick the row with the lowest MAE
best_name = candidates["R2"].idxmax()
print(f"Best model based on R2 : {best_name}")

# map model names to the actual fitted pipelines
trained = {
    "Linear Regression": linreg,
    "Decision Tree": tree,
    "Random Forest": rf,
    "Random Forest (slow tuning)": best_model,
}
final_model = trained[best_name]

# save the whole pipeline (preprocess + model) in one file
joblib.dump(final_model, "model.joblib", compress=5)
print("Model saved to model.joblib")

***

## ✅ **Conclusion :**

***

### **What we did :**

1. **Loaded** the cleaned dataset produced by the EDA notebook.
2. **Defined** the target (`Average delay of all trains at arrival`) and selected 11 features (4 categorical + 7 numeric).
3. **Split** the data into 80% training / 20% test, with a fixed `random_state` to make every model comparable.
4. **Built a preprocessing pipeline** with one-hot encoding for the strings, stored in a `ColumnTransformer` and reused inside every model `Pipeline`.
5. **Trained 4 models** a baseline (predict the mean), a Linear Regression, a single Decision Tree and a Random Forest.
6. **Compared** the models with MAE, RMSE and R² — the Random Forest is the best.
7. **Tuned** the Random Forest with `GridSearchCV` and 5-fold cross-validation, we trained in two different ways : a quick one (8 combinations, ~1 min) and a slow one (960 combinations, ~hours). The best hyperparameters of each run are hardcoded in the notebook to avoid spending hours every time.
8. **Compared quick vs slow tuning** to show why it's worse spending more time at hyperparameter search.
9. **Compared before vs after tuning** to show the real improvement brought by the tuned model.
10. **Saved** the best pipeline (preprocessing + tuned Random Forest) into `model.joblib` for the Streamlit dashboard.

***

### **Why is the Random forest a good model for the project :**

* **It beats the baseline by a wide margin** (R² goes from 0 to around 0.83).
* **It is plug-and-play** for the dashboard : the whole pipeline is in a single `model.joblib` file.